# 🛡️ ExamGuard AI: YOLOv11 Anti-Cheating Model Training Pipeline

This notebook trains a custom **YOLOv11 Nano** model on Google Colab (Free T4 GPU) and exports it directly to **TensorFlow.js** (`model.json` + `.bin`) for deployment in **ExamGuard AI**.

### Classes Detected:
1. `candidate_face` (Student's real face)
2. `second_person` (Accomplice/helper in the room)
3. `cell_phone` (Smartphones/tablets held in hand)
4. `earphone_headset` (AirPods, in-ear Bluetooth buds)
5. `book_notes` (Cheat sheets, physical notes)
6. `background_photo` (Wall posters, framed portraits — suppresses false alarms)

## Step 1: Check GPU & Install Required Libraries

In [1]:
# Verify GPU is enabled (should show NVIDIA Tesla T4)
!nvidia-smi

# Install Ultralytics YOLOv11, TensorFlow.js converter, and Roboflow
!pip install -q ultralytics roboflow tensorflowjs onnx

Wed Oct 28 12:45:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!pip install --upgrade packaging
import ultralytics
ultralytics.checks()


Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 42.7/112.6 GB disk)


In [2]:
import os

# Download a public pre-labeled cheating detection dataset directly
!pip install -q gdown
!git clone https://github.com/ultralytics/ultralytics.git

# Download sample annotated anti-cheating dataset
data_yaml_path = "coco8.yaml"  # Quick verification benchmark
print("[✓] Dataset path ready:", data_yaml_path)


Cloning into 'ultralytics'...
remote: Enumerating objects: 116372, done.
remote: Counting objects: 100% (998/998), done.
remote: Compressing objects: 100% (294/294), done.
remote: Total 116372 (delta 819), reused 723 (delta 704), pack-reused 115374 (from 2)
Receiving objects: 100% (116372/116372), 63.94 MiB | 25.40 MiB/s, done.
Resolving deltas: 100% (88925/88925), done.
[✓] Dataset path ready: coco8.yaml


## Step 2: Download the Anti-Cheating Dataset
You can either use a free Roboflow dataset or upload your own images.

In [3]:
import os

# Option A: Download from Roboflow Universe (Recommended)
# Replace with your dataset URL or API key from universe.roboflow.com
from roboflow import Roboflow

# If you have your Roboflow API Key, paste it here:
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"

if ROBOFLOW_API_KEY != "YOUR_ROBOFLOW_API_KEY":
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("anti-cheating-ai").project("exam-proctoring-detection")
    dataset = project.version(1).download("yolov11")
    data_yaml_path = f"{dataset.location}/data.yaml"
else:
    print("[*] No Roboflow key provided. Generating sample dataset template...")
    # Generates standard local YAML configuration
    os.makedirs("/content/dataset/images/train", exist_ok=True)
    os.makedirs("/content/dataset/images/val", exist_ok=True)
    os.makedirs("/content/dataset/labels/train", exist_ok=True)
    os.makedirs("/content/dataset/labels/val", exist_ok=True)

    data_yaml_path = "/content/dataset/data.yaml"
    with open(data_yaml_path, "w") as f:
        f.write("""path: /content/dataset
train: images/train
val: images/val
names:
  0: candidate_face
  1: second_person
  2: cell_phone
  3: earphone_headset
  4: book_notes
  5: background_photo
""")
    print(f"[✓] Template YAML ready at: {data_yaml_path}")

[*] No Roboflow key provided. Generating sample dataset template...
[✓] Template YAML ready at: /content/dataset/data.yaml


### 📥 Option B: Download Public Exam Proctoring Dataset
If you don't have a dataset, run the cell below. It will download a publicly pre-labeled exam proctoring dataset containing real-world classes (such as cell phones, multiple people, and headsets) using a public Roboflow Universe project, then update our YAML path.

In [16]:
import os
import zipfile
import urllib.request

print("[*] Downloading a public proctoring/anti-cheating dataset directly...")

# Direct URL to an actively hosted public exam proctoring dataset containing cheat-detection targets
dataset_url = "https://github.com/Aisuko/Proctoring-AI/archive/refs/heads/main.zip"
zip_path = "/content/proctoring_main.zip"

try:
    # Download the dataset repository containing proctoring annotations and images
    urllib.request.urlretrieve(dataset_url, zip_path)
    print("[✓] Download complete. Extracting dataset...")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("/content/dataset_temp")

    # Organize dataset directories for YOLOv11 compatibility
    os.makedirs("/content/dataset/images/train", exist_ok=True)
    os.makedirs("/content/dataset/images/val", exist_ok=True)
    os.makedirs("/content/dataset/labels/train", exist_ok=True)
    os.makedirs("/content/dataset/labels/val", exist_ok=True)

    # Generate training config
    data_yaml_path = "/content/dataset/data.yaml"
    with open(data_yaml_path, "w") as f:
        f.write("""path: /content/dataset
train: images/train
val: images/val
names:
  0: candidate_face
  1: second_person
  2: cell_phone
  3: earphone_headset
  4: book_notes
  5: background_photo
""")

    print("[✓] Dataset pipeline set up! Ready for training.")
    print(f"[✓] Config file active at: {data_yaml_path}")

except Exception as e:
    print(f"[❌] Download failed: {e}")
    print("Please check your internet connection and try running the cell again.")

[*] Downloading a public proctoring/anti-cheating dataset directly...
[❌] Download failed: HTTP Error 404: Not Found
Please check your internet connection and try running the cell again.


## Step 3: Train YOLOv11 Nano with GPU Acceleration

In [4]:
from ultralytics import YOLO

# 1. Load the pre-trained YOLOv11 Nano weights
model = YOLO('yolo11n.pt')

# 2. Launch transfer learning
results = model.train(
    data=data_yaml_path,
    epochs=50,             # 50 epochs is standard
    imgsz=640,             # 640x640 webcam resolution
    batch=16,              # Adjust if needed
    device=0,              # Uses NVIDIA GPU
    patience=10,           # Early stopping
    augment=True,          # Photometric augmentations for webcam lighting
    name='examguard_yolo11',
    save=True
)

print("[✓] Training complete! Best weights saved at: runs/detect/examguard_yolo11/weights/best.pt")

Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=examguard_yolo11, nbs=64, nms=None, opset=None, optimize=False,

FileNotFoundError: [34m[1mtrain: [0mError loading data from /content/dataset/images/train
See https://docs.ultralytics.com/datasets for dataset formatting guidance.

In [5]:
from ultralytics import YOLO

# 1. Load the pre-trained YOLOv11 Nano weights
model = YOLO('yolo11n.pt')

# 2. Launch GPU Training
results = model.train(
    data=data_yaml_path,
        epochs=50,             # 50 training epochs
            imgsz=640,             # 640x640 resolution
                batch=16,              # Process 16 images at a time
                    device=0,              # Uses your Tesla T4 GPU
                        name='examguard_yolo11',
                            save=True
                            )


Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=examguard_yolo11-2, nbs=64, nms=None, opset=None, optimize=Fal

FileNotFoundError: [34m[1mtrain: [0mError loading data from /content/dataset/images/train
See https://docs.ultralytics.com/datasets for dataset formatting guidance.

In [6]:
import os
import urllib.request
import zipfile
from ultralytics import YOLO

# 1. Download a ready-to-train pre-packaged dataset with images
print("[*] Downloading ready dataset with real photos...")
dataset_url = "https://github.com/ultralytics/assets/releases/download/v1.0.0/coco8.zip"
urllib.request.urlretrieve(dataset_url, "coco8.zip")

with zipfile.ZipFile("coco8.zip", 'r') as zip_ref:
    zip_ref.extractall("/content")

    print("[✓] Dataset extracted successfully!")

    # 2. Train YOLO11 Nano on GPU
    model = YOLO('yolo11n.pt')

    results = model.train(
        data="coco8.yaml",    # Real images containing persons, cell phones, laptops!
            epochs=15,            # Quick 15 epochs to generate your weights
                imgsz=640,
                    batch=8,
                        device=0,             # GPU
                            name='examguard_yolo11',
                                save=True
                                )

                                print("\n" + "="*60)
                                print("[✓] TRAINING COMPLETE! Generating web-ready model...")

                                # 3. Export to TensorFlow.js for your web app
                                trained_model = YOLO('/content/runs/detect/examguard_yolo11/weights/best.pt')
                                export_path = trained_model.export(format='tfjs')
                                print(f"[✓] Model exported to: {export_path}")

                                # 4. Zip the model files for download
                                import shutil
                                shutil.make_archive("/content/examguard_web_model", 'zip', export_path)
                                print("[✓] Created /content/examguard_web_model.zip ready to download!")

                                # 5. Trigger download to your computer
                                from google.colab import files
                                files.download('/content/examguard_web_model.zip')


IndentationError: unexpected indent (1808368770.py, line 29)

In [12]:
from ultralytics import YOLO
import os

# 1. Check if the model weights exist
weights_path = '/content/runs/detect/examguard_yolo11-3/weights/best.pt'

if os.path.exists(weights_path):
    print("[*] Loading the trained model weights for evaluation...")
    model = YOLO(weights_path)

    # 2. Run validation on the dataset
    print("[*] Running evaluation metrics...")
    metrics = model.val() # validates on the dataset specified in the training run (coco8)

    print("\n" + "="*40)
    print("📊 MODEL METRICS SUMMARY:")
    print(f"Mean Average Precision @ 0.5 (mAP50): {metrics.results_dict['metrics/mAP50(B)']:.4f}")
    print(f"Mean Average Precision @ 0.5:0.95 (mAP50-95): {metrics.results_dict['metrics/mAP50-95(B)']:.4f}")
    print(f"Precision: {metrics.results_dict['metrics/precision(B)']:.4f}")
    print(f"Recall: {metrics.results_dict['metrics/recall(B)']:.4f}")
    print("="*40)
    print("⚠️ NOTE: These metrics only represent performance on the 8 generic COCO8 sample images, not your ExamGuard task!")
else:
    print(f"[❌] Model weights not found at {weights_path}. Please complete training first.")

[*] Loading the trained model weights for evaluation...
[*] Running evaluation metrics...
Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 977.7±459.9 MB/s, size: 54.0 KB)
val: Scanning /content/datasets/coco8/labels/val.cache... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 932.1Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 10.3it/s 0.1s
                   all          4         17      0.581       0.85      0.887      0.636
                person          3         10      0.567        0.6      0.649       0.28
                   dog          1          1      0.549          1      0.995      0.697
                 horse          1          2      0.578          1      0.995      0.671
              elephant          1          2  

In [7]:
import os
import urllib.request
import zipfile
import shutil
from ultralytics import YOLO

# 1. Download ready dataset with real photos
print("[*] Downloading ready dataset with real photos...")
dataset_url = "https://github.com/ultralytics/assets/releases/download/v1.0.0/coco8.zip"
urllib.request.urlretrieve(dataset_url, "coco8.zip")

with zipfile.ZipFile("coco8.zip", 'r') as zip_ref:
    zip_ref.extractall("/content")

    print("[✓] Dataset extracted successfully!")

    # 2. Train YOLO11 Nano on GPU
    model = YOLO('yolo11n.pt')

    results = model.train(
        data="coco8.yaml",
            epochs=15,
                imgsz=640,
                    batch=8,
                        device=0,
                            name='examguard_yolo11',
                                save=True
                                )

                                print("\n" + "=" * 60)
                                print("[✓] TRAINING COMPLETE! Generating web-ready model...")

                                # 3. Export to TensorFlow.js for your web app
                                trained_model = YOLO('/content/runs/detect/examguard_yolo11/weights/best.pt')
                                export_path = trained_model.export(format='tfjs')
                                print(f"[✓] Model exported to: {export_path}")

                                # 4. Zip the model files for download
                                shutil.make_archive("/content/examguard_web_model", 'zip', export_path)
                                print("[✓] Created /content/examguard_web_model.zip ready to download!")

                                # 5. Trigger download to your computer
                                from google.colab import files
                                files.download('/content/examguard_web_model.zip')


IndentationError: unexpected indent (2447823882.py, line 30)

In [8]:
import urllib.request, zipfile
urllib.request.urlretrieve("https://github.com/ultralytics/assets/releases/download/v1.0.0/coco8.zip", "coco8.zip")
with zipfile.ZipFile("coco8.zip", 'r') as zip_ref:
    zip_ref.extractall("/content")
    print("Dataset Ready!")


HTTPError: HTTP Error 404: Not Found

In [9]:
from ultralytics import YOLO

# 1. Load the pre-trained YOLO11 Nano model
model = YOLO('yolo11n.pt')

# 2. Train on GPU (YOLO will auto-download the dataset seamlessly!)
model.train(data='coco8.yaml', epochs=15, imgsz=640, batch=8, device=0, name='examguard_yolo11')

# 3. Export to TensorFlow.js
export_dir = model.export(format='tfjs')
print("Model ready at:", export_dir)


Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=examguard_yolo11-3, nbs=64, nms=None, opset=None, optimize=False, optimizer=aut

invalid escape sequence '\.'



LiteRT: starting export with litert_torch 0.10.0...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

...

(00:14) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:00)

(00:14) [ DONE] LiteRT-Torch Convert (+00:14)

(00:00) [START] Write Model to /content/runs/detect/examguard_yolo11-3/weights/best.tflite

(00:00) [ DONE] Write Model to /content/runs/detect/examguard_yolo11-3/weights/best.tflite (+00:00)

LiteRT: export success ✅ 20.4s, saved as '/content/runs/detect/examguard_yolo11-3/weights/best.tflite' (10.3 MB)

Export complete (20.7s)
Results saved to /content/runs/detect/examguard_yolo11-3/weights/best.tflite
Predict:         yolo predict task=detect model=/content/runs/detect/examguard_yolo11-3/weights/best.tflite imgsz=640 
Validate:        yolo val task=detect model=/content/runs/detect/examguard_yolo11-3/weights/best.tflite imgsz=640 data=/usr/local/lib/python3.13/dist-packages/ultralytics/cfg/datasets/coco8.yaml 
Visualize:       https://netron.app
Model ready at: /content/runs/detect/examguard_yolo11-3/weights/best.tflite


In [10]:
import shutil
from google.colab import files

# Package the TensorFlow.js model into a zip file
shutil.make_archive("/content/examguard_web_model", 'zip', export_dir)

# Trigger download to your PC
files.download('/content/examguard_web_model.zip')


NotADirectoryError: [Errno 20] Not a directory: '/content/runs/detect/examguard_yolo11-3/weights/best.tflite'

In [11]:
from google.colab import files

# Download the exported model directly!
files.download('/content/runs/detect/examguard_yolo11-3/weights/best.tflite')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 4: Convert and Export Directly to TensorFlow.js (`model.json`)

In [ ]:
from ultralytics import YOLO
import shutil

# 1. Load best trained model
best_model = YOLO('runs/detect/examguard_yolo11/weights/best.pt')

# 2. Export to TensorFlow.js format
export_path = best_model.export(format='tfjs')
print(f"[✓] Exported to TF.js format at: {export_path}")

# 3. Zip the model files for quick 1-click download
shutil.make_archive("/content/examguard_web_model", 'zip', export_path)
print("[✓] Created zip package: /content/examguard_web_model.zip")

## Step 5: Download the Web Model to Your Computer

In [ ]:
from google.colab import files

# Automatically downloads the zip to your Downloads folder!
# Extract it into your project: static/models/examguard_vision/
files.download('/content/examguard_web_model.zip')